# Studying the effect of pooling - Design C (one repetition datasets, mixing observers)

Designs A (scaled-down pool) and B (crowd-like pool) compare pooled data with the individual observers at equal size.

Design C compares the same trials answered by one observer or by the five observers, using a single repeat. It compares dimensionality and triplet accuracy. 

A one-repetition dataset joins one session of trial set 1 and one session of trial set 2 (600 trials), and assigns to every triplet 
either the answers of a single observer (*single obs*), or an answer of a random pick of the five observers (*mix obs*), balanced across observers.

In [1]:
import ast
import itertools
import os

import numpy as np
import pandas as pd
import sklearn

import cblearn

from load_data import load_all
from dimensionality_estimation import estimate_dimensionality
from io_json import read_json_dataclass, save_json_dataclass
from utils import read_dimensionality
from size_controlled_pooling import one_rep_dataset, to_triplets

# seeds the cross-validation splits and the SOE initialisations of every fit;
# draw k of mix obs is seeded with SEED + k
SEED = 0
# draws of mix obs per combination
N_DRAWS = 5
# the combinations (i, j), 0-based: the i-th session of trial set 1 with the j-th of trial set 2
COMBINATIONS = list(itertools.product(range(3), repeat=2))
LABEL_MIX = 'design-C-mix-obs'
LABEL_SINGLE = 'design-C-single-obs'
# all fitted runs; read back instead of refitting when it and the jsons exist
RUNS_CSV = '../data/found-dimensionality-embeddings-design-C-draws.csv'
# one row per finished fit while the fits run, so that an interrupted run resumes; deleted once
# RUNS_CSV is written
PARTIAL_CSV = '../data/found-dimensionality-embeddings-design-C-draws.partial.csv'

OBSERVERS = ['O1', 'O2', 'O3', 'O4', 'O5']
TASKS = ['classic', 'odd one out']
IMAGE_SETS = ['action', 'greebles', 'taxonomic']

print('cblearn:', cblearn.__version__)
print('sklearn:', sklearn.__version__)

cblearn: 0.3.0
sklearn: 1.5.2


# Reading data

`load_all(fill_missing_O_by_P=True)` relabels the sessions that O5 did under the pseudonym P1 (1a, 1b, 2a, 2b) as O5, and those of P2 as O6.
Keeping O1 - O5 gives, for every task and image set, 30 sessions: every observer ran each of the two trial sets (300 triplets each) in three sessions (repeats). The cell checks this, and that sorting the sessions of an observer and trial set by date gives the order of their labels, so the recording order used below is unambiguous.

In [2]:
triplet_df = load_all(fill_missing_O_by_P=True)
triplet_df = triplet_df[triplet_df['observer'].isin(OBSERVERS)]

for (obs, task, image_set, triplet_set), sessions in triplet_df.groupby(['observer', 'task', 'image_set',
                                                                         'triplet_set']):
    assert len(sessions) == 3
    assert list(sessions.sort_values('date')['session']) == sorted(sessions['session'])

# sessions with fewer than 300 trials: timeouts, and three sessions stopped after 150 trials
#n_trials = triplet_df.assign(n=triplet_df['responses'].apply(len))
#n_trials[n_trials['n'] < 300][['observer', 'task', 'image_set', 'session', 'n', 'n_timeout']].sort_values('n')

# Design

For every task and image set there were two trial sets, of 300 triplets each, and every observer answered each trial set in three sessions, one per repeat. One session held one trial set, 300 trials.

Here we simulate pooling at this amount of data, 600 trials.
A one-repetition dataset joins one session of trial set 1 with one session of trial set 2, holding 600 trials,and thus being a third of one observer's data (1800 trials). 
The sessions of each trial set are numbered 1 to 3 in the order in which they were recorded, and a combination $(i, j)$ takes the $i$-th session of trial set 1 and the $j$-th session of trial set 2. We constructed datasets using all 3 x 3 = 9 combinations.

For every combination we build two kinds of dataset:

- *Single obs.* The two sessions of one observer, as recorded. This gives 5 observers x 9 combinations = 45 datasets per task and image set.
- *Mix obs.* The 300 triplets of each trial set are split at random into five groups of 60, one per observer, and every triplet takes the answer that its observer gave in that observer's own session of the combination (function `one_rep_dataset`). Every observer thus answers 2 x 60 = 120 of the 600 triplets. Each combination is drawn `N_DRAWS = 5` times, which gives 9 x 5 = 45 datasets per task and image set.

For a given combination, the single obs and mix obs datasets have the same size and the same triplets, and come from the same sessions; they differ only in who answered each triplet. A question asked in two layouts counts as two triplets: in single obs the same observer answers both, in mix obs mostly two different observers.

A triplet that the assigned observer did not answer in its session (a timeout) is kept as missing in both kinds of dataset, and is not replaced.

Draw $k$ of a combination is seeded with `SEED + k` together with the task, the image set and combination's indices. This is made so that the seed is unique for every draw and it does not depend on which other datasets are built. All fits use the same seeds for the cross-validation splits and the SOE initialisations.

### Comparisons

- *Mix obs vs single obs.* The two have the same size, triplets and sessions. A difference between them is therefore the effect of mixing observers.
- *Dimensionality and triplet accuracy.* Mix obs and single obs are compared by their dimensionality and by their cross-validated triplet accuracy at the estimated dimensionality.

In [3]:
runs_records = []  # one row per dataset: the data that is fitted
for task in TASKS:
    for image_set in IMAGE_SETS:
        condition = triplet_df[(triplet_df['task'] == task) & (triplet_df['image_set'] == image_set)]
        for i, j in COMBINATIONS:
            combination = f'{i + 1}-{j + 1}'
            datasets = [(f'{LABEL_SINGLE}-{obs}', 'single', obs, [obs], 0,
                         f'{LABEL_SINGLE}-{obs}-comb{i + 1}{j + 1}') for obs in OBSERVERS]
            datasets += [(LABEL_MIX, 'mix', ' '.join(OBSERVERS), OBSERVERS, draw,
                          f'{LABEL_MIX}-comb{i + 1}{j + 1}-draw{draw}') for draw in range(N_DRAWS)]
            for label, variant, answered_by, observers, draw, fit in datasets:
                # a single observer answers all rows, so its generator draws nothing that matters
                rng = np.random.default_rng([SEED + draw, TASKS.index(task), IMAGE_SETS.index(image_set), i, j])
                record = one_rep_dataset(condition, observers, (i, j), rng)
                runs_records.append({'observer': label, 'variant': variant, 'answered_by': answered_by,
                                     'combination': combination, 'draw': draw, 'seed': SEED + draw, 'fit': fit,
                                     'task': task, 'image_set': image_set,
                                     'n_trials': len(record['responses']), **record,
                                     'embedding_ready_triplets': to_triplets(task, record['raw_triplets'],
                                                                             record['responses'])})

runs_df = pd.DataFrame(runs_records)
runs_df['n_triplets'] = runs_df['embedding_ready_triplets'].apply(len)
assert (runs_df['n_triplets'] == runs_df['n_trials'] * np.where(runs_df['task'] == 'classic', 1, 2)).all()
print(len(runs_df), 'datasets:', runs_df['variant'].value_counts().to_dict())

540 datasets: {'single': 270, 'mix': 270}


In [4]:
# 2 tasks x 3 image sets = 6 conditions
# 6 conditions x 45 combinations = 270 datasets 

In [5]:
display(runs_df.groupby(['task', 'image_set', 'variant'])['n_trials'].describe()[['count', 'min', 'mean', 'max']])

count    min        mean    max
task        image_set variant                                 
classic     action    mix       45.0  562.0  589.755556  600.0
                      single    45.0  450.0  589.733333  600.0
            greebles  mix       45.0  562.0  589.600000  600.0
                      single    45.0  450.0  589.266667  600.0
            taxonomic mix       45.0  561.0  589.066667  600.0
                      single    45.0  449.0  589.466667  600.0
odd one out action    mix       45.0  598.0  599.777778  600.0
                      single    45.0  598.0  599.800000  600.0
            greebles  mix       45.0  599.0  599.888889  600.0
                      single    45.0  599.0  599.866667  600.0
            taxonomic mix       45.0  599.0  599.777778  600.0
                      single    45.0  599.0  599.800000  600.0

Trials per dataset (at most 600), for every task, image set and variant, and the number of triplets every observer answers in draw 0 of mix obs for combination 1-1 (120 each, 60 per trial set; fewer where the observer timed out).

In [6]:
example = runs_df[(runs_df['variant'] == 'mix') & (runs_df['combination'] == '1-1') & (runs_df['draw'] == 0)]
pd.DataFrame([{'task': row.task, 'image_set': row.image_set, 'trials': row.n_trials,
               **pd.Series(row.source_observer).value_counts().reindex(OBSERVERS).to_dict()}
              for row in example.itertuples()])

,task,image_set,trials,O1,O2,O3,O4,O5
0,classic,action,571,120,120,120,91,120
1,classic,greebles,571,120,120,120,92,119
2,classic,taxonomic,566,120,120,120,87,119
3,odd one out,action,600,120,120,120,120,120
4,odd one out,greebles,600,120,120,120,120,120
5,odd one out,taxonomic,599,120,120,120,119,120


# Estimating dimensionality of every dataset

Each dataset is fitted with the significance criterion of `cblearn`. The cross-validation results of every fit are saved in `../data/triplet_accuracy_vs_dimensionality-{fit}-{task}-{image_set}.json`, with `fit` equal to `design-C-single-obs-{observer}-comb{i}{j}` or `design-C-mix-obs-comb{i}{j}-draw{k}`. The embedding is fitted at the dimensionality of the significance criterion.

When the runs table (`RUNS_CSV`) and the json of every fit exist, the cell reads them back instead of fitting, after checking that the rebuilt datasets equal the saved ones and that the table agrees with the jsons.

Otherwise the fits run, and every finished fit is appended at once to `PARTIAL_CSV`, with its triplets, its dimensionality and its embedding. A rerun after an interruption skips a fit only when it has a row in `PARTIAL_CSV` and a json, and its saved triplets equal the rebuilt ones; if they differ, the cell stops, because the saved fits come from other data. When all fits are done, `RUNS_CSV` is written and `PARTIAL_CSV` deleted. To refit everything, delete `RUNS_CSV`, `PARTIAL_CSV` and the jsons.

In [7]:
# the fits run when the runs table or the json of any fit is missing; otherwise they are read
# back: the dimensionalities from the jsons, the embeddings from the table
json_names = [f"../data/triplet_accuracy_vs_dimensionality-{d['fit']}-{d['task']}-{d['image_set']}.json"
              for _, d in runs_df.iterrows()]
loaded = os.path.exists(RUNS_CSV) and all(os.path.exists(name) for name in json_names)
keys = ['fit', 'task', 'image_set']


def dimensionality_of(name):
    """The estimated dimensionality of a fit, read from its json."""
    return read_json_dataclass(name).estimated_dimension


if loaded:
    saved = pd.read_csv(RUNS_CSV)
    assert list(saved[keys].itertuples(index=False)) == list(runs_df[keys].itertuples(index=False)), \
        'the saved runs are in another order'
    # the datasets rebuilt above must be the ones that were fitted
    assert all(np.array_equal(np.array(ast.literal_eval(s)), t)
               for s, t in zip(saved['embedding_ready_triplets'], runs_df['embedding_ready_triplets'])), \
        'the rebuilt datasets differ from the saved ones'
    dimensionality = [dimensionality_of(name) for name in json_names]
    assert dimensionality == list(saved['dimensionality'])
    embedding = [ast.literal_eval(s) for s in saved['embedding']]
    print(f'read {len(json_names)} saved fits and {RUNS_CSV}')
else:
    # fits finished by an earlier, interrupted run
    done = {}
    if os.path.exists(PARTIAL_CSV):
        done = {tuple(row[k] for k in keys): row for _, row in pd.read_csv(PARTIAL_CSV).iterrows()}
    dimensionality, embedding = [], []

    for (ix, d), name in zip(runs_df.iterrows(), json_names):
        key = tuple(d[k] for k in keys)
        triplets = d['embedding_ready_triplets']

        if key in done and os.path.exists(name):
            row = done[key]
            if not np.array_equal(np.array(ast.literal_eval(row['embedding_ready_triplets'])), triplets):
                raise ValueError(f'{key}: the saved fit in {PARTIAL_CSV} was made on other triplets; '
                                 'delete it and the jsons of design C to refit')
            dim = dimensionality_of(name)
            assert dim == row['dimensionality']
            fitted_embedding = np.array(ast.literal_eval(row['embedding']))
            status = 'read'
        else:
            dim_result, fitted_embedding = estimate_dimensionality(triplets, random_state=SEED,
                                                                   soe_random_state=SEED)
            # saving triplet accuracy and statistics
            save_json_dataclass(dim_result, name)
            dim = dimensionality_of(name)
            # one row per finished fit, written at once so that an interrupted run keeps it
            pd.DataFrame([{**dict(zip(keys, key)), 'embedding_ready_triplets': np.asarray(triplets).tolist(),
                           'dimensionality': dim,
                           'embedding': fitted_embedding.tolist()}]).to_csv(
                PARTIAL_CSV, mode='a', header=not os.path.exists(PARTIAL_CSV), index=False)
            status = 'fitted'
        assert fitted_embedding.shape[1] == dim

        dimensionality.append(dim)
        embedding.append(fitted_embedding.tolist())

        print(f"{status}: {d['fit']} - {d['task']} - {d['image_set']} - {len(triplets)} triplets: "
              f"{dim} dimension(s)")

runs_df['dimensionality'] = list(dimensionality)
runs_df['embedding'] = embedding

read 540 saved fits and ../data/found-dimensionality-embeddings-design-C-draws.csv


In [8]:
if not loaded:
    # all datasets; list-valued columns are saved as their repr, read back with ast.literal_eval
    to_save = runs_df.copy()
    for column in ['raw_triplets', 'responses', 'triplet_index', 'triplet_set', 'source_observer',
                   'source_session', 'embedding_ready_triplets', 'embedding']:
        to_save[column] = to_save[column].apply(lambda x: np.asarray(x).tolist())
    to_save.to_csv(RUNS_CSV, index=False)
    # every fit is now in RUNS_CSV
    if os.path.exists(PARTIAL_CSV):
        os.remove(PARTIAL_CSV)

# Summary

For each task and image set: the dimensionality of every observer (all their data) and their mean, of pooled-5x, and the mean dimensionality over the 45 datasets of single obs and over the 45 datasets of mix obs.

In [9]:
# the individual observers and pooled-5x are read from their jsons (read_dimensionality), the source
# of the reported values
mean_c = runs_df.groupby(['variant', 'task', 'image_set'])['dimensionality'].mean()

rows = []
for task in TASKS:
    for image_set in IMAGE_SETS:
        row = {'task': task, 'image_set': image_set}
        for obs in OBSERVERS:
            row[obs] = read_dimensionality(obs, task, image_set)
        row['mean O1 - O5'] = np.mean([row[obs] for obs in OBSERVERS])
        row['pooled-5x'] = read_dimensionality('pooled O1 - O5', task, image_set)
        row['single obs, mean of 45'] = mean_c['single', task, image_set]
        row['mix obs, mean of 45'] = mean_c['mix', task, image_set]
        rows.append(row)

summary = pd.DataFrame(rows)
with pd.option_context('display.precision', 1):
    display(summary)

,task,image_set,O1,O2,O3,O4,O5,mean O1 - O5,pooled-5x,"single obs, mean of 45","mix obs, mean of 45"
0,classic,action,2,1,2,2,1,1.6,4,1.1,1.0
1,classic,greebles,2,2,1,2,3,2.0,2,1.5,1.5
2,classic,taxonomic,3,2,2,1,2,2.0,2,1.2,1.1
3,odd one out,action,2,1,3,2,1,1.8,3,1.2,1.0
4,odd one out,greebles,1,2,1,2,2,1.6,2,1.1,1.3
5,odd one out,taxonomic,2,2,1,1,1,1.4,2,1.2,1.0


Dimensionality per observer under the significance criterion. For every image set and task, column `one rep` is the mean over the 9 combinations of the observer's single obs one-rep datasets (one session of each trial list, about 600 trials, a third of the observer's data), column `all reps` the dimensionality of the observer with all its data, the three repeats of both trial lists (about 1800 trials; `read_dimensionality`). Below the observers: their mean, and the pooled data, with mix obs (mean over its 45 one-rep datasets) under `one rep` and pooled-5x under `all reps`.

In [10]:
columns = pd.MultiIndex.from_tuples([(image_set, task, kind) for image_set in IMAGE_SETS for task in TASKS
                                     for kind in ['one rep', 'all reps']])
per_observer = pd.DataFrame(index=OBSERVERS + ['mean O1 - O5', 'pooled O1 - O5'], columns=columns, dtype=float)
for task in TASKS:
    for image_set in IMAGE_SETS:
        condition = runs_df[(runs_df['task'] == task) & (runs_df['image_set'] == image_set)]
        for obs in OBSERVERS:
            per_observer.loc[obs, (image_set, task, 'one rep')] = condition.loc[
                (condition['variant'] == 'single') & (condition['answered_by'] == obs), 'dimensionality'].mean()
            per_observer.loc[obs, (image_set, task, 'all reps')] = read_dimensionality(obs, task, image_set, 'significance')
        for kind in ['one rep', 'all reps']:
            per_observer.loc['mean O1 - O5', (image_set, task, kind)] = per_observer.loc[OBSERVERS, (image_set, task, kind)].mean()
        # pooled data at the two sizes: mix obs under one rep, pooled-5x under all reps
        per_observer.loc['pooled O1 - O5', (image_set, task, 'one rep')] = condition.loc[condition['variant'] == 'mix', 'dimensionality'].mean()
        per_observer.loc['pooled O1 - O5', (image_set, task, 'all reps')] = read_dimensionality('pooled O1 - O5', task, image_set,
                                                                                   'significance')
per_observer.round(1)

action                               greebles           \
               classic          odd one out           classic            
               one rep all reps     one rep all reps  one rep all reps   
O1                 1.0      2.0         1.0      2.0      1.8      2.0   
O2                 1.0      1.0         1.0      1.0      2.3      2.0   
O3                 1.2      2.0         1.9      3.0      1.0      1.0   
O4                 1.2      2.0         1.1      2.0      1.2      2.0   
O5                 1.0      1.0         1.0      1.0      1.0      3.0   
mean O1 - O5       1.1      1.6         1.2      1.8      1.5      2.0   
pooled O1 - O5     1.0      4.0         1.0      3.0      1.5      2.0   

                                    taxonomic                                
               odd one out            classic          odd one out           
                   one rep all reps   one rep all reps     one rep all reps  
O1                     1.0      1.0       1.4      3.0         1.8      2.0  
O2                     1.0      2.0       1.1      2.0         1.4      2.0  
O3                     1.2      1.0       1.2      2.0         1.0      1.0  
O4                     1.0      2.0       1.0      1.0         1.0      1.0  
O5                     1.3      2.0       1.3      2.0         1.0      1.0  
mean O1 - O5           1.1      1.6       1.2      2.0         1.2      1.4  
pooled O1 - O5         1.3      2.0       1.1      2.0         1.0      2.0

Test accuracy per observer, in the layout of the dimensionality table above: the cross-validated test accuracy (%, mean over the 150 folds of a fit) at the dimensionality that the fit estimated under the significance criterion. Column `one rep` is the mean over the observer's 9 single obs one-rep datasets, column `all reps` the accuracy of the observer's fit to all its data (Step 1 json). Below the observers: their mean, and the pooled data, with mix obs (mean over its 45 one-rep datasets) under `one rep` and pooled-5x under `all reps`.

In [11]:
def accuracy_at_estimate(name):
    """Test accuracy (%) of a fit at its estimated dimensionality, mean over its folds."""
    dim_result = read_json_dataclass(name)
    return 100 * np.mean(dim_result.test_scores[dim_result.estimated_dimension - 1])


runs_df['accuracy_at_estimate'] = [accuracy_at_estimate(name) for name in json_names]
step1_json = '../data/triplet_accuracy_vs_dimensionality-{obs}-{task}-{image_set}.json'

accuracy_per_observer = pd.DataFrame(index=OBSERVERS + ['mean O1 - O5', 'pooled O1 - O5'], columns=columns, dtype=float)
for task in TASKS:
    for image_set in IMAGE_SETS:
        condition = runs_df[(runs_df['task'] == task) & (runs_df['image_set'] == image_set)]
        for obs in OBSERVERS:
            accuracy_per_observer.loc[obs, (image_set, task, 'one rep')] = condition.loc[
                (condition['variant'] == 'single') & (condition['answered_by'] == obs), 'accuracy_at_estimate'].mean()
            accuracy_per_observer.loc[obs, (image_set, task, 'all reps')] = accuracy_at_estimate(
                step1_json.format(obs=obs, task=task, image_set=image_set))
        for kind in ['one rep', 'all reps']:
            accuracy_per_observer.loc['mean O1 - O5', (image_set, task, kind)] = \
                accuracy_per_observer.loc[OBSERVERS, (image_set, task, kind)].mean()
        # pooled data at the two sizes: mix obs under one rep, pooled-5x under all reps
        accuracy_per_observer.loc['pooled O1 - O5', (image_set, task, 'one rep')] = condition.loc[
            condition['variant'] == 'mix', 'accuracy_at_estimate'].mean()
        accuracy_per_observer.loc['pooled O1 - O5', (image_set, task, 'all reps')] = accuracy_at_estimate(
            step1_json.format(obs='pooled O1 - O5', task=task, image_set=image_set))
accuracy_per_observer.round(1)

action                               greebles           \
               classic          odd one out           classic            
               one rep all reps     one rep all reps  one rep all reps   
O1                84.5     87.1        95.1     96.3     80.6     82.1   
O2                85.6     86.8        96.6     96.1     88.4     89.9   
O3                76.4     80.3        81.3     83.6     68.6     69.3   
O4                78.0     80.4        90.1     91.8     72.7     77.3   
O5                89.6     91.2        98.9     99.4     76.7     81.6   
mean O1 - O5      82.9     85.1        92.4     93.4     77.4     80.0   
pooled O1 - O5    81.4     85.6        87.9     89.7     70.7     75.5   

                                    taxonomic                                
               odd one out            classic          odd one out           
                   one rep all reps   one rep all reps     one rep all reps  
O1                    89.5     90.0      81.5     85.6        95.0     95.7  
O2                    97.7     98.4      90.1     93.2        96.0     97.3  
O3                    77.0     76.7      84.1     86.2        91.5     91.3  
O4                    88.7     89.9      75.5     75.8        94.9     95.1  
O5                    89.1     90.2      91.0     94.7        99.4     99.7  
mean O1 - O5          88.4     89.0      84.5     87.1        95.4     95.8  
pooled O1 - O5        79.8     82.0      79.7     83.5        90.8     91.8